# 第75章 综合项目与作品集

用 plotly 内置的 experiment 数据集（100 名被试，control/treatment 两组，3 个结果指标，含性别标签）完整走一遍 A/B 实验评估流程：随机化校验 → 效应估计 → 置换检验 → 多重比较校正 → 亚组与交互 → 功效复盘 → 上线决策。

## 项目背景

数据来源：plotly 内置 experiment 数据集，随 plotly 包一起分发，常用于统计检验教学与示例，100 行、5 列。,字段含义：group 为随机分组（control 对照 / treatment 实验），experiment_1/2/3 为三个结果指标（分数型，越高越好），gender 为被试性别。,业务设定：假设这是一次产品改版实验，三个指标分别是「主指标：任务完成分」「护栏指标：满意度分」「探索指标：参与度分」。,关于数据真实性：该数据集为 plotly 官方随包分发的标准示例数据，取值为已发布的固定数值（非本课程用随机数生成），可复现、可对照；但它不是某次真实商业实验的原始日志，因此结论只用于演示方法，不代表真实产品结论。

## 学习目标

- 先验证随机化是否成功，再看结果——顺序颠倒会让整个实验失去可信度
- 掌握「点估计 + 置信区间 + 效应量 + p 值」四件套，理解为什么单看 p 值不够
- 用 numpy 手写置换检验和自举，不依赖 scipy 也能做出严谨推断
- 理解多重比较问题：测 3 个指标就有 3 次犯错机会，必须做 Holm 校正
- 识别亚组分析的陷阱，区分「真实交互」与「数据挖掘出的假象」
- 用 MDE 与功效复盘回答「这个实验本来能检出多大的效应」


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| group | 分组 | control / treatment，随机分配 |
| gender | 性别 | 亚组分析维度 |
| experiment_1 | 主指标（任务完成分） | 决定是否上线的唯一指标 |
| experiment_2 | 护栏指标（满意度分） | 不允许显著下降 |
| experiment_3 | 探索指标（参与度分） | 仅用于生成假设，不作决策依据 |
| diff | 组间差值 | = treatment 均值 - control 均值 |
| cohen_d | 效应量 | = 差值 / 合并标准差，衡量差异的实际大小 |
| p_perm | 置换检验 p 值 | 在「无效应」假设下观测到当前差异的概率 |

## 数据质量检查清单

- 两组样本量是否接近（严重失衡提示分流实现有 bug）
- 协变量（性别）在两组间分布是否一致——这是随机化是否成功的核心证据
- 是否有缺失值与重复被试（同一人进两组会污染独立性假设）
- 各指标分布是否存在极端值或截断（分数被封顶会压缩效应）


## 项目任务

1. 载入数据并完成随机化校验（样本量、协变量平衡、缺失、重复）
2. 对三个指标做描述性对比，输出均值、中位数、标准差、差值
3. 可视化组间分布，用箱线图 + 抖动散点同时展示汇总与个体
4. 手写置换检验计算 p 值，并算 Cohen's d 效应量
5. 用自举法给组间差值加置信区间
6. 做 Holm 多重比较校正，说明校正前后的结论差异
7. 按性别做亚组与交互分析，检查是否存在符号反转
8. 做 MDE 与功效复盘，给出明确的上线 / 不上线决策


## 步骤1｜载入数据并做随机化校验

A/B 实验的第一步永远不是看结果，而是验证随机化。如果两组在实验开始前就不可比，后面所有的差异都无法归因到干预。这一步叫 A/A 校验或平衡性检验。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

import plotly.express as px

# plotly 随包分发的标准实验数据集
exp = px.data.experiment()

METRICS = ["experiment_1", "experiment_2", "experiment_3"]
LABELS = {
    "experiment_1": "主指标 任务完成分",
    "experiment_2": "护栏指标 满意度分",
    "experiment_3": "探索指标 参与度分",
}

print("=" * 92)
print("数据集: plotly experiment | A/B 实验结果")
print("=" * 92)
print(f"形状: {exp.shape}")
print(exp.head(5).to_string(index=False))
print("\n数据类型:")
print(exp.dtypes.to_string())

print("\n" + "=" * 92)
print("随机化校验（这一步不通过，后面结果全部作废）")
print("=" * 92)

# 1. 样本量平衡
cnt = exp["group"].value_counts()
n_c, n_t = int(cnt.get("control", 0)), int(cnt.get("treatment", 0))
ratio = n_t / n_c if n_c else np.nan
print(f"1. 样本量  control={n_c}  treatment={n_t}  比例={ratio:.2f}")
print(f"   判定: {'通过（0.9~1.1 区间内）' if 0.9 <= ratio <= 1.1 else '关注（分流可能不均）'}")

# 2. 协变量平衡：性别分布
ct = pd.crosstab(exp["group"], exp["gender"])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
print("\n2. 协变量平衡（性别构成 %）")
print(ct_pct.round(1).to_string())
gap = float(abs(ct_pct.iloc[0] - ct_pct.iloc[1]).max())
print(f"   最大构成差异: {gap:.1f} 个百分点")
print(f"   判定: {'通过' if gap < 10 else '关注（协变量不平衡，需在分析中控制）'}")

# 3. 缺失与重复
print(f"\n3. 缺失值总数: {int(exp.isna().sum().sum())}  完全重复行: {int(exp.duplicated().sum())}")

# 4. 指标分布形态
print("\n4. 各指标分布（检查极端值与截断）")
desc = exp[METRICS].describe().T[["min", "25%", "50%", "75%", "max", "std"]]
desc["偏度"] = [exp[m].skew() for m in METRICS]
print(desc.round(2).to_string())
print("\n   说明: 若 max 处堆积大量样本说明指标被封顶，会压缩可观测效应。")


## 步骤2｜描述性对比：先看清差异有多大

推断之前先描述。均值告诉你平均效应，中位数告诉你典型用户，标准差告诉你个体差异有多大。相对提升比绝对差值更容易跨指标比较，但基数小的时候相对值会失真，两个都要给。


In [ ]:
rows = []
for m in METRICS:
    c = exp.loc[exp["group"] == "control", m].to_numpy()
    t = exp.loc[exp["group"] == "treatment", m].to_numpy()
    diff = t.mean() - c.mean()
    rows.append({
        "指标": LABELS[m],
        "control均值": c.mean(),
        "treatment均值": t.mean(),
        "绝对差": diff,
        "相对提升%": diff / c.mean() * 100 if c.mean() else np.nan,
        "control中位": np.median(c),
        "treatment中位": np.median(t),
        "合并标准差": np.sqrt(((len(c)-1)*c.var(ddof=1) + (len(t)-1)*t.var(ddof=1)) / (len(c)+len(t)-2)),
    })

summary = pd.DataFrame(rows).set_index("指标")
print("=" * 92)
print("组间描述性对比")
print("=" * 92)
print(summary.round(3).to_string())

print("\n" + "-" * 92)
print("初步判断")
print("-" * 92)
for m in METRICS:
    r = summary.loc[LABELS[m]]
    direction = "上升" if r["绝对差"] > 0 else "下降"
    # 差值与个体差异的量级对比
    scale = abs(r["绝对差"]) / r["合并标准差"]
    print(f"{LABELS[m]:<18} {direction} {abs(r['绝对差']):.2f} 分 "
          f"({r['相对提升%']:+.1f}%)，约为个体标准差的 {scale:.2f} 倍")

print("\n关键提醒: 差值只有个体标准差的零点几倍时，")
print("          意味着组间差异远小于组内个体差异，这种效应很容易被随机波动模仿。")
print("          所以必须做假设检验，不能直接下结论。")


## 步骤3｜分布可视化：箱线图 + 抖动散点

只画均值柱状图是 A/B 报告最常见的错误——它把分布压成一个点，看不出重叠程度。箱线图给分位数，抖动散点给每个个体，两者叠加才能看出「两组到底有没有分开」。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
rng = np.random.default_rng(0)  # 仅用于散点抖动的横向位置，不生成任何分析数据
colors = {"control": "#8da0cb", "treatment": "#fc8d62"}

for ax, m in zip(axes, METRICS):
    groups = ["control", "treatment"]
    data = [exp.loc[exp["group"] == g, m].to_numpy() for g in groups]

    bp = ax.boxplot(data, positions=[1, 2], widths=0.5, patch_artist=True,
                    showmeans=True, meanline=True)
    for patch, g in zip(bp["boxes"], groups):
        patch.set_facecolor(colors[g])
        patch.set_alpha(0.45)

    for i, (g, d) in enumerate(zip(groups, data), start=1):
        x = i + rng.uniform(-0.13, 0.13, size=len(d))
        ax.scatter(x, d, s=16, color=colors[g], edgecolor="white", linewidth=0.4,
                   alpha=0.85, zorder=3)
        ax.scatter([i], [d.mean()], marker="D", s=60, color="black", zorder=4)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["对照组", "实验组"])
    ax.set_title(LABELS[m], fontsize=11)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("得分")
fig.suptitle("组间分布对比（箱=分位数，点=个体，黑菱形=均值）", fontsize=13)
plt.tight_layout()
plt.show()

print("读图要点:")
print("1. 两组箱体重叠范围越大，说明区分度越低，越需要靠检验而非肉眼判断")
print("2. 均值（菱形）和中位数（箱内横线）位置不一致，提示分布偏斜")
print("3. 抖动散点暴露了样本量：每组仅约 50 人，任何结论都要考虑抽样误差")
for m in METRICS:
    c = exp.loc[exp["group"] == "control", m]
    t = exp.loc[exp["group"] == "treatment", m]
    lo = max(c.min(), t.min()); hi = min(c.max(), t.max())
    span = max(c.max(), t.max()) - min(c.min(), t.min())
    print(f"   {LABELS[m]:<18} 取值区间重叠度: {(hi-lo)/span*100:.0f}%")


## 步骤4｜置换检验与效应量（不依赖 scipy）

置换检验的逻辑很直接：如果干预真的无效，那么组标签就是随意贴的。把标签打乱几千次，看「随机贴标签能不能造出当前这么大的差异」。它不要求正态分布，小样本下比 t 检验更稳。效应量 Cohen's d 回答另一个问题：差异在实际意义上算大还是小。


In [ ]:
def perm_test(a, b, n_perm=10000, seed=42):
    """双侧置换检验：返回观测差值与 p 值。"""
    a, b = np.asarray(a, float), np.asarray(b, float)
    obs = b.mean() - a.mean()
    pool = np.concatenate([a, b])
    n_a = len(a)
    rg = np.random.default_rng(seed)
    cnt = 0
    for _ in range(n_perm):
        rg.shuffle(pool)
        if abs(pool[n_a:].mean() - pool[:n_a].mean()) >= abs(obs) - 1e-12:
            cnt += 1
    return obs, (cnt + 1) / (n_perm + 1)   # +1 平滑，避免 p=0


def cohen_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    n1, n2 = len(a), len(b)
    s = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (b.mean() - a.mean()) / s if s > 0 else np.nan


def d_label(d):
    a = abs(d)
    if a < 0.2: return "可忽略"
    if a < 0.5: return "小"
    if a < 0.8: return "中"
    return "大"


N_PERM = 10000
res = []
for m in METRICS:
    c = exp.loc[exp["group"] == "control", m].to_numpy()
    t = exp.loc[exp["group"] == "treatment", m].to_numpy()
    obs, p = perm_test(c, t, n_perm=N_PERM)
    d = cohen_d(c, t)
    res.append({"指标": LABELS[m], "metric": m, "差值": obs,
                "p_perm": p, "cohen_d": d, "效应量": d_label(d)})

test = pd.DataFrame(res).set_index("指标")
print("=" * 92)
print(f"置换检验结果（{N_PERM} 次重排，双侧）")
print("=" * 92)
print(test[["差值", "p_perm", "cohen_d", "效应量"]].round(4).to_string())

print("\n" + "-" * 92)
print("单指标判定（未做多重比较校正，alpha=0.05）")
print("-" * 92)
for idx, r in test.iterrows():
    sig = "显著" if r["p_perm"] < 0.05 else "不显著"
    print(f"{idx:<18} p={r['p_perm']:.4f} {sig:<6} d={r['cohen_d']:+.3f}（{r['效应量']}）")

print("\n方法说明:")
print("1. 置换检验只假设「组标签可交换」，不假设正态分布，适合小样本")
print("2. p 值加 1 平滑，保证 p > 0——0 次超越不等于概率为零")
print("3. p 值小 ≠ 效应大：p 受样本量影响，d 不受，两者必须一起看")


## 步骤5｜自举置信区间：给效应加上不确定性

p 值只回答是非题，置信区间回答「效应可能有多大」。区间跨过 0 说明连方向都没定；区间很宽说明样本量不足。业务决策看的是区间下界——最坏情况下还赚不赚。


In [ ]:
def boot_diff_ci(a, b, n_boot=5000, alpha=0.05, seed=7):
    """对 (b均值 - a均值) 做百分位自举置信区间。"""
    a, b = np.asarray(a, float), np.asarray(b, float)
    rg = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        sa = rg.choice(a, size=len(a), replace=True)
        sb = rg.choice(b, size=len(b), replace=True)
        diffs[i] = sb.mean() - sa.mean()
    lo, hi = np.percentile(diffs, [alpha / 2 * 100, (1 - alpha / 2) * 100])
    return lo, hi, diffs


ci_rows, boot_store = [], {}
for m in METRICS:
    c = exp.loc[exp["group"] == "control", m].to_numpy()
    t = exp.loc[exp["group"] == "treatment", m].to_numpy()
    lo, hi, diffs = boot_diff_ci(c, t)
    boot_store[m] = diffs
    ci_rows.append({
        "指标": LABELS[m], "metric": m,
        "差值": t.mean() - c.mean(), "CI下界": lo, "CI上界": hi,
        "区间宽度": hi - lo,
        "跨过0": "是" if lo < 0 < hi else "否",
        "P(效应>0)": (diffs > 0).mean(),
    })

ci = pd.DataFrame(ci_rows).set_index("指标")
print("=" * 92)
print("自举 95% 置信区间（5000 次重抽样）")
print("=" * 92)
print(ci[["差值", "CI下界", "CI上界", "区间宽度", "跨过0", "P(效应>0)"]].round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 4.2))
ypos = np.arange(len(METRICS))
for i, m in enumerate(METRICS):
    r = ci.loc[LABELS[m]]
    ax.plot([r["CI下界"], r["CI上界"]], [i, i], color="#555", lw=2.4)
    ax.plot([r["CI下界"], r["CI上界"]], [i, i], "|", color="#555", ms=12)
    ax.scatter([r["差值"]], [i], s=90, color="#fc8d62", zorder=3, edgecolor="k", linewidth=0.6)

ax.axvline(0, color="crimson", ls="--", lw=1.2)
ax.set_yticks(ypos)
ax.set_yticklabels([LABELS[m] for m in METRICS])
ax.set_xlabel("treatment 均值 - control 均值")
ax.set_title("效应量点估计与 95% 自举置信区间", fontsize=12)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("解读规则:")
print("1. 区间跨过红色 0 线 → 无法排除「无效应」，不能声称有提升")
print("2. 区间越宽 → 样本量越不足，估计越不可靠")
print("3. 决策要看下界：下界为负说明存在真实变差的可能性")


## 步骤6｜多重比较校正：测 3 个指标就有 3 次犯错机会

每做一次检验就有 5% 概率误报。测 3 个指标，至少一次误报的概率升到约 14%。Holm 方法把 p 值从小到大排序，逐个用递减的阈值比较，既控制了整体错误率，又比 Bonferroni 更有检出力。


In [ ]:
alpha = 0.05
k = len(METRICS)
fwer_naive = 1 - (1 - alpha) ** k
print("=" * 92)
print("为什么需要校正")
print("=" * 92)
print(f"单次检验犯第一类错误概率: {alpha:.0%}")
print(f"独立做 {k} 次检验，至少一次误报的概率: {fwer_naive:.1%}")
print(f"也就是说，即使改版完全无效，也有约 {fwer_naive:.0%} 的概率至少看到一个「显著」指标。")


def holm(pvals, alpha=0.05):
    """Holm-Bonferroni 逐步校正，返回 (调整后p, 是否拒绝) 的原序数组。"""
    p = np.asarray(pvals, float)
    n = len(p)
    order = np.argsort(p)
    adj_sorted = np.empty(n)
    running = 0.0
    for i, idx in enumerate(order):
        val = (n - i) * p[idx]
        running = max(running, val)          # 保证单调不减
        adj_sorted[i] = min(running, 1.0)
    adj = np.empty(n)
    adj[order] = adj_sorted
    return adj, adj < alpha


pvals = test["p_perm"].to_numpy()
adj, reject = holm(pvals)

cmp = pd.DataFrame({
    "指标": test.index,
    "原始p": pvals,
    "Bonferroni p": np.minimum(pvals * k, 1.0),
    "Holm 调整p": adj,
    "原始判定": np.where(pvals < alpha, "显著", "不显著"),
    "Holm判定": np.where(reject, "显著", "不显著"),
    "cohen_d": test["cohen_d"].to_numpy(),
}).set_index("指标")

print("\n" + "=" * 92)
print("校正前后对比")
print("=" * 92)
print(cmp.round(4).to_string())

flipped = cmp[(cmp["原始判定"] == "显著") & (cmp["Holm判定"] == "不显著")]
print("\n" + "-" * 92)
if len(flipped):
    print(f"校正后有 {len(flipped)} 个指标从「显著」变为「不显著」:")
    for idx in flipped.index:
        print(f"  {idx}: p={cmp.loc[idx,'原始p']:.4f} -> Holm={cmp.loc[idx,'Holm 调整p']:.4f}")
    print("这类结论在多指标实验里最危险——它们通常是多重比较的产物，不是真实效应。")
else:
    print("校正前后判定一致，结论对多重比较不敏感（稳健）。")

print("\n实践规范: 实验开始前就锁定唯一主指标，护栏指标只用于否决，")
print("          探索指标不参与决策，只用于生成下一次实验的假设。")


## 步骤7｜亚组与交互分析：最容易造假的一步

亚组分析的诱惑是：整体不显著时，切分人群总能找到一个「显著」的格子。但切得越细，误报越多。正确做法是把亚组结论当假设而非结论，并检查是否存在符号反转（辛普森悖论）。


In [ ]:
MAIN = "experiment_1"
print("=" * 92)
print(f"按性别拆分主指标: {LABELS[MAIN]}")
print("=" * 92)

sub_rows = []
for g, gd in exp.groupby("gender"):
    c = gd.loc[gd["group"] == "control", MAIN].to_numpy()
    t = gd.loc[gd["group"] == "treatment", MAIN].to_numpy()
    if len(c) < 5 or len(t) < 5:
        continue
    obs, p = perm_test(c, t, n_perm=5000, seed=11)
    sub_rows.append({
        "亚组": g, "n_control": len(c), "n_treatment": len(t),
        "control均值": c.mean(), "treatment均值": t.mean(),
        "差值": obs, "p_perm": p, "cohen_d": cohen_d(c, t),
    })

sub = pd.DataFrame(sub_rows).set_index("亚组")
print(sub.round(3).to_string())

overall_diff = float(test.loc[LABELS[MAIN], "差值"])
print(f"\n整体差值: {overall_diff:+.3f}")

signs = np.sign(sub["差值"].to_numpy())
print("\n" + "-" * 92)
print("符号一致性检查")
print("-" * 92)
if len(set(signs)) > 1:
    print("发现符号反转：不同亚组的效应方向相反 —— 存在辛普森悖论风险。")
    for idx, r in sub.iterrows():
        print(f"  {idx}: {r['差值']:+.3f}")
    print("此时整体均值差是各亚组效应的加权平均，权重由亚组样本量决定，")
    print("若两组的亚组构成不同，整体结论可能纯粹由构成差异驱动。")
else:
    print(f"所有亚组效应方向一致（{'均为正' if signs[0] > 0 else '均为负'}），未见符号反转。")

# 交互效应：差值之差
if len(sub) == 2:
    inter = float(sub["差值"].iloc[1] - sub["差值"].iloc[0])
    print(f"\n交互效应（两亚组差值之差）: {inter:+.3f}")
    print(f"相对整体效应的量级: {abs(inter) / (abs(overall_diff) + 1e-9):.1f} 倍")

# 亚组检验的多重比较代价
n_sub_tests = len(sub) * len(METRICS)
print(f"\n多重比较代价: 3 指标 x {len(sub)} 亚组 = {n_sub_tests} 次检验，")
print(f"未校正时至少一次误报概率约 {1 - 0.95 ** n_sub_tests:.0%}。")

fig, ax = plt.subplots(figsize=(8.5, 4.4))
x = np.arange(len(sub))
w = 0.36
ax.bar(x - w/2, sub["control均值"], w, label="对照组", color="#8da0cb")
ax.bar(x + w/2, sub["treatment均值"], w, label="实验组", color="#fc8d62")
for i, (idx, r) in enumerate(sub.iterrows()):
    ax.text(i, max(r["control均值"], r["treatment均值"]) * 1.02,
            f"{r['差值']:+.2f}\np={r['p_perm']:.3f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(sub.index)
ax.set_ylabel("主指标均值")
ax.set_title(f"亚组效应对比｜{LABELS[MAIN]}", fontsize=12)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("\n规范: 亚组结论只能作为下一次实验的假设，")
print("      要确认必须做预注册的独立验证实验，样本量按该亚组重新估算。")


## 步骤8｜功效复盘与上线决策

实验结束后要回答一个关键问题：如果真有效应，这个样本量本来能不能检出来？MDE（最小可检测效应）给出答案。若 MDE 远大于业务上有意义的提升幅度，那么「不显著」只说明实验没做够，不说明改版无效。


In [ ]:
Z_A, Z_B = 1.959964, 0.841621   # alpha=0.05 双侧, power=0.8

print("=" * 92)
print("功效复盘：本次实验的检出能力")
print("=" * 92)
power_rows = []
for m in METRICS:
    c = exp.loc[exp["group"] == "control", m].to_numpy()
    t = exp.loc[exp["group"] == "treatment", m].to_numpy()
    n = min(len(c), len(t))
    sd = np.sqrt(((len(c)-1)*c.var(ddof=1) + (len(t)-1)*t.var(ddof=1)) / (len(c)+len(t)-2))
    mde_abs = (Z_A + Z_B) * sd * np.sqrt(2 / n)
    obs = t.mean() - c.mean()
    power_rows.append({
        "指标": LABELS[m], "每组n": n, "合并sd": sd,
        "MDE绝对值": mde_abs,
        "MDE相对%": mde_abs / c.mean() * 100 if c.mean() else np.nan,
        "实测差值": obs,
        "实测/MDE": abs(obs) / mde_abs if mde_abs else np.nan,
    })

pw = pd.DataFrame(power_rows).set_index("指标")
print(pw.round(3).to_string())

print("\n解读: 「实测/MDE」< 1 表示观测效应小于本实验的检出门槛，")
print("      此时不显著属于「证据不足」，而不是「已证明无效」。")

TARGET_LIFT = 0.05   # 业务认为 5% 提升才值得上线
print("\n" + "-" * 92)
print(f"若业务目标提升为 {TARGET_LIFT:.0%}，各指标所需样本量")
print("-" * 92)
for m in METRICS:
    c = exp.loc[exp["group"] == "control", m].to_numpy()
    t = exp.loc[exp["group"] == "treatment", m].to_numpy()
    sd = np.sqrt(((len(c)-1)*c.var(ddof=1) + (len(t)-1)*t.var(ddof=1)) / (len(c)+len(t)-2))
    delta = c.mean() * TARGET_LIFT
    need = int(np.ceil(2 * ((Z_A + Z_B) * sd / delta) ** 2)) if delta > 0 else -1
    print(f"{LABELS[m]:<18} 每组需 {need:>6} 人（当前 {min(len(c), len(t))} 人，"
          f"缺口 {max(0, need - min(len(c), len(t))):>6} 人）")

print("\n" + "=" * 92)
print("上线决策")
print("=" * 92)
main_row = cmp.loc[LABELS[MAIN]]
main_ci = ci.loc[LABELS[MAIN]]
guard_row = cmp.loc[LABELS["experiment_2"]]
guard_ci = ci.loc[LABELS["experiment_2"]]

main_ok = bool(main_row["Holm判定"] == "显著" and main_ci["CI下界"] > 0)
guard_bad = bool(guard_row["Holm判定"] == "显著" and guard_ci["CI上界"] < 0)

print(f"主指标  {LABELS[MAIN]}: 差值 {main_ci['差值']:+.3f}, "
      f"95%CI [{main_ci['CI下界']:.3f}, {main_ci['CI上界']:.3f}], "
      f"Holm p={main_row['Holm 调整p']:.4f}, d={main_row['cohen_d']:+.3f}")
print(f"护栏指标 {LABELS['experiment_2']}: 差值 {guard_ci['差值']:+.3f}, "
      f"95%CI [{guard_ci['CI下界']:.3f}, {guard_ci['CI上界']:.3f}], "
      f"Holm p={guard_row['Holm 调整p']:.4f}")

if guard_bad:
    decision, why = "不上线", "护栏指标显著下降，无论主指标表现如何都应否决"
elif main_ok:
    decision, why = "上线", "主指标在多重比较校正后仍显著为正，且护栏未被击穿"
else:
    decision, why = "不上线（证据不足）", "主指标未通过校正后检验，置信区间跨过 0"

print(f"\n决策: {decision}")
print(f"依据: {why}")

print("\n后续动作:")
acts = [
    f"按上表补足样本量至可检出 {TARGET_LIFT:.0%} 提升的规模后重跑实验",
    "预注册唯一主指标与分析方案，避免事后挑指标",
    "亚组差异作为假设进入下一轮预注册实验，不作为本次决策依据",
    "补充实验期间的分流日志监控，确认无样本比不匹配（SRM）问题",
]
for i, a in enumerate(acts, 1):
    print(f"{i}. {a}")

print("\n分析局限: 数据为 plotly 随包分发的示例数据集（固定数值、可复现），")
print("          用于演示评估流程；每组约 50 人的规模在真实业务实验中偏小。")


## 结论与表达

- 随机化校验必须先于结果分析：样本量比例、协变量构成、缺失与重复四项通过后，组间差异才可能归因到干预本身。
- 「点估计 + 置信区间 + 效应量 + p 值」四件套缺一不可——p 值只回答是非，置信区间给出效应范围，Cohen's d 判断实际意义。
- 置换检验与自举都能用 numpy 十几行写出来，不依赖 scipy，且不要求正态假设，在每组约 50 人的小样本下比 t 检验更稳。
- 多重比较是多指标实验的头号陷阱：3 个指标的整体误报率约 14%，Holm 校正会推翻部分「显著」结论，这类结论通常是统计噪声。
- MDE 复盘把「不显著」区分成两种情况：真无效应，还是样本量不足导致的证据不足——两者的后续动作完全不同。


## 项目验收清单

- 能独立完成随机化校验四项检查，并说明协变量不平衡为什么会毁掉实验
- 能手写置换检验与自举置信区间，并解释两者分别回答什么问题
- 能实现 Holm 校正并说明它与 Bonferroni 的差别
- 能在亚组分析中识别符号反转，并说明为什么亚组结论只能当假设
- 能用 MDE 判断一次不显著的实验是「无效应」还是「样本不足」

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

用 plotly 内置的 experiment 数据集（100 名被试，control/treatment 两组，3 个结果指标，含性别标签）完整走一遍 A/B 实验评估流程：随机化校验 → 效应估计 → 置换检验 → 多重比较校正 → 亚组与交互 → 功效复盘 → 上线决策。


### 你已经完成

- 先验证随机化是否成功，再看结果——顺序颠倒会让整个实验失去可信度
- 掌握「点估计 + 置信区间 + 效应量 + p 值」四件套，理解为什么单看 p 值不够
- 用 numpy 手写置换检验和自举，不依赖 scipy 也能做出严谨推断
- 理解多重比较问题：测 3 个指标就有 3 次犯错机会，必须做 Holm 校正
- 识别亚组分析的陷阱，区分「真实交互」与「数据挖掘出的假象」
- 用 MDE 与功效复盘回答「这个实验本来能检出多大的效应」


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 载入数据并完成随机化校验（样本量、协变量平衡、缺失、重复） |
| 步骤 2 | 对三个指标做描述性对比，输出均值、中位数、标准差、差值 |
| 步骤 3 | 可视化组间分布，用箱线图 + 抖动散点同时展示汇总与个体 |
| 步骤 4 | 手写置换检验计算 p 值，并算 Cohen's d 效应量 |
| 步骤 5 | 用自举法给组间差值加置信区间 |
| 步骤 6 | 做 Holm 多重比较校正，说明校正前后的结论差异 |
| 步骤 7 | 按性别做亚组与交互分析，检查是否存在符号反转 |
| 步骤 8 | 做 MDE 与功效复盘，给出明确的上线 / 不上线决策 |


### 质量与结论提醒

- 两组样本量是否接近（严重失衡提示分流实现有 bug）
- 协变量（性别）在两组间分布是否一致——这是随机化是否成功的核心证据
- 是否有缺失值与重复被试（同一人进两组会污染独立性假设）
- 随机化校验必须先于结果分析：样本量比例、协变量构成、缺失与重复四项通过后，组间差异才可能归因到干预本身。
- 「点估计 + 置信区间 + 效应量 + p 值」四件套缺一不可——p 值只回答是非，置信区间给出效应范围，Cohen's d 判断实际意义。
- 置换检验与自举都能用 numpy 十几行写出来，不依赖 scipy，且不要求正态假设，在每组约 50 人的小样本下比 t 检验更稳。
- 多重比较是多指标实验的头号陷阱：3 个指标的整体误报率约 14%，Holm 校正会推翻部分「显著」结论，这类结论通常是统计噪声。
- MDE 复盘把「不显著」区分成两种情况：真无效应，还是样本量不足导致的证据不足——两者的后续动作完全不同。


### 项目交付检查

- [ ] 能独立完成随机化校验四项检查，并说明协变量不平衡为什么会毁掉实验
- [ ] 能手写置换检验与自举置信区间，并解释两者分别回答什么问题
- [ ] 能实现 Holm 校正并说明它与 Bonferroni 的差别
- [ ] 能在亚组分析中识别符号反转，并说明为什么亚组结论只能当假设
- [ ] 能用 MDE 判断一次不显著的实验是「无效应」还是「样本不足」
